# Explainable SIEM Alert Prioritization

Prioritize a noisy alert stream and show analysts why each alert received a high score.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Train a logistic ranking model, calculate recall within a fixed review budget, and expose per-feature contributions.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def sigmoid(values):
    clipped = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped))

def split_indices(size, test_fraction=0.25):
    shuffled = rng.permutation(size)
    split_at = int(size * (1 - test_fraction))
    return shuffled[:split_at], shuffled[split_at:]

def standardize(train_values, test_values):
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0)
    std = np.where(std < 1e-9, 1.0, std)
    return (train_values - mean) / std, (test_values - mean) / std, mean, std

def fit_logistic(features, labels, steps=1400, learning_rate=0.08, l2=0.01):
    design = np.column_stack([np.ones(len(features)), features])
    weights = np.zeros(design.shape[1])
    for _ in range(steps):
        probabilities = sigmoid(design @ weights)
        gradient = design.T @ (probabilities - labels) / len(labels)
        gradient[1:] += l2 * weights[1:]
        weights -= learning_rate * gradient
    return weights

def predict_probability(features, weights):
    design = np.column_stack([np.ones(len(features)), features])
    return sigmoid(design @ weights)

def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    accuracy = (tp + tn) / max(len(labels), 1)
    return pd.Series({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    })


## Steps

### 1. Generate synthetic SIEM alerts


In [2]:
alert_count = 2200
severity = rng.integers(1, 6, alert_count)
asset_criticality = rng.integers(1, 6, alert_count)
detection_confidence = rng.beta(2.2, 2.0, alert_count)
correlated_alerts = rng.poisson(2.2, alert_count)
privileged_identity = rng.binomial(1, 0.12, alert_count)
internet_exposed = rng.binomial(1, 0.20, alert_count)
age_minutes = rng.exponential(75, alert_count).clip(0, 720)

incident_probability = sigmoid(
    -6.3
    + 0.52 * severity
    + 0.44 * asset_criticality
    + 2.5 * detection_confidence
    + 0.20 * correlated_alerts
    + 0.85 * privileged_identity
    + 0.65 * internet_exposed
    - 0.002 * age_minutes
)
confirmed_incident = rng.binomial(1, incident_probability)

alerts = pd.DataFrame({
    "severity": severity,
    "asset_criticality": asset_criticality,
    "detection_confidence": detection_confidence,
    "correlated_alerts": correlated_alerts,
    "privileged_identity": privileged_identity,
    "internet_exposed": internet_exposed,
    "age_minutes": age_minutes,
    "confirmed_incident": confirmed_incident,
})

print("Alert count:", len(alerts))
print("Confirmed-incident rate:", round(alerts["confirmed_incident"].mean(), 3))
print(alerts.head(6).round(3).to_string(index=False))


Alert count: 2200
Confirmed-incident rate: 0.226
 severity  asset_criticality  detection_confidence  correlated_alerts  privileged_identity  internet_exposed  age_minutes  confirmed_incident
        1                  2                 0.321                  1                    0                 0      219.632                   0
        4                  1                 0.466                  1                    0                 0       56.377                   0
        4                  1                 0.238                  2                    0                 0        0.086                   0
        3                  2                 0.761                  3                    0                 0        3.299                   0
        3                  3                 0.415                  1                    0                 0       31.882                   0
        5                  4                 0.499                  1                    0         

### 2. Train, rank, and explain alerts


In [3]:
feature_names = [column for column in alerts.columns if column != "confirmed_incident"]
train_rows, test_rows = split_indices(len(alerts))
train_values = alerts.loc[train_rows, feature_names].to_numpy(float)
test_values = alerts.loc[test_rows, feature_names].to_numpy(float)
train_labels = alerts.loc[train_rows, "confirmed_incident"].to_numpy(int)
test_labels = alerts.loc[test_rows, "confirmed_incident"].to_numpy(int)

train_scaled, test_scaled, feature_mean, feature_std = standardize(train_values, test_values)
siem_weights = fit_logistic(train_scaled, train_labels)
incident_probability_test = predict_probability(test_scaled, siem_weights)
incident_prediction = (incident_probability_test >= 0.5).astype(int)
siem_metrics = classification_metrics(test_labels, incident_prediction)

ranked_alerts = alerts.loc[test_rows].copy()
ranked_alerts["priority_score"] = incident_probability_test
ranked_alerts = ranked_alerts.sort_values("priority_score", ascending=False)
review_count = max(1, int(len(ranked_alerts) * 0.10))
total_incidents = ranked_alerts["confirmed_incident"].sum()
top_decile_recall = ranked_alerts.head(review_count)["confirmed_incident"].sum() / max(total_incidents, 1)

contribution_values = test_scaled * siem_weights[1:]
top_position = int(np.argmax(incident_probability_test))
contribution_table = pd.DataFrame({
    "feature": feature_names,
    "contribution": contribution_values[top_position],
}).sort_values("contribution", ascending=False)

print("Classification metrics:")
print(siem_metrics.round(3).to_string())
print("\nRecall captured in top 10% of alerts:", round(top_decile_recall, 3))
print("\nExplanation for the highest-priority alert:")
print(contribution_table.round(3).to_string(index=False))
print("\nPriority queue sample:")
print(ranked_alerts.head(8).round(3).to_string(index=False))


Classification metrics:
accuracy       0.785
precision      0.558
recall         0.234
f1             0.330
tp            29.000
fp            23.000
tn           403.000
fn            95.000

Recall captured in top 10% of alerts: 0.25

Explanation for the highest-priority alert:
             feature  contribution
            severity         0.917
   asset_criticality         0.772
detection_confidence         0.664
    internet_exposed         0.512
   correlated_alerts         0.297
 privileged_identity        -0.080
         age_minutes        -0.126

Priority queue sample:
 severity  asset_criticality  detection_confidence  correlated_alerts  privileged_identity  internet_exposed  age_minutes  confirmed_incident  priority_score
        5                  5                 0.755                  4                    0                 1      137.742                   1           0.806
        5                  5                 0.738                  2                    0         

## Checks


In [4]:
assert 0.03 < alerts["confirmed_incident"].mean() < 0.60
assert top_decile_recall >= 0.20
assert ranked_alerts["priority_score"].between(0, 1).all()
assert ranked_alerts["priority_score"].is_monotonic_decreasing
print("Checks passed: plausible alert balance, useful review-budget recall, and explainable sorted scores.")


Checks passed: plausible alert balance, useful review-budget recall, and explainable sorted scores.


## Next Steps

        - Optimize the review budget against analyst staffing and incident impact.
- Add rule family, tactic, technique, and historical disposition features.
- Monitor score calibration and false-negative severity over time.
